# Guess a recurrence with Hologuess

Start with exact values of a sequence or Taylor coefficients of a function. Hologuess searches for an operator that vanishes on those observations.

From the checkout, run `sage -pip install -e .`. Open this notebook with a **SageMath kernel** that has `ore_algebra` installed. [Installation details](../README.md#install-and-test)

## Fibonacci numbers

In Ore algebra, `Sn` names the forward shift $E f(n)=f(n+1)$. The LaTeX display writes this shift as $S$.

The list below gives $F_0,\ldots,F_{15}$. At order two, each window is a triple $(F_n,F_{n+1},F_{n+2})$. Setting `holdout=4` reserves the final four windows for an exact check.

- **Guess** using the windows starting at $n=0,\ldots,9$. Together they use $F_0,\ldots,F_{11}$.
- **Check** the returned recurrence on the windows starting at $n=10,\ldots,13$. These windows reach the four withheld values $F_{12},\ldots,F_{15}$.

In [1]:
%display latex
from hologuess import guess, guess_qsequence
from ore_algebra import OreAlgebra

values = [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610]
shift = OreAlgebra(QQ["n"], "Sn")
fibonacci_guess = guess(values, shift, order=2, degree=0, holdout=4)
fibonacci_guess.operator

Sn^2 - Sn - 1

The operator gives $f(n+2)-f(n+1)-f(n)=0$ on the checked windows. Here `order=2` allows shifts through order two and `degree=0` asks for constant coefficients.

The fitting prefix comes from SymPy's [Fibonacci recurrence test](https://github.com/sympy/sympy/blob/16fa855354eb7bcabd3fe10993841e03b1382692/sympy/concrete/tests/test_guess.py#L19-L25).

## Taylor coefficients

To guess a differential equation, give the ordinary Taylor coefficients $[x^k]f(x)$. For $f(x)=e^x$ these are $1/k!$. The LaTeX display writes the derivative as $D$. Its Ore name is `Dx`.

In [2]:
coefficients = [1 / factorial(k) for k in range(12)]
differential = OreAlgebra(QQ["x"], "Dx")
taylor_guess = guess(coefficients, differential, order=1, degree=0)
taylor_guess.operator

Dx - 1

The operator $D-1$ gives $f'(x)-f(x)=0$ on the available Taylor coefficients. This input also appears in Hologuess's [backend regression](../tests/test_flint_backend.py).

## Change or copy the LaTeX notation

Choose the LaTeX symbols for this kernel. The setting also applies to operators you have already guessed.

In [3]:
from hologuess import operator_latex, set_latex_symbols

P = fibonacci_guess.operator
L = taylor_guess.operator
set_latex_symbols(S=r"\mathcal{S}", D=r"\partial_x")
display(P, L)

Sn^2 - Sn - 1

Dx - 1

Print a LaTeX string to copy into a paper. The `S` argument below applies to this string alone. `print(L)` still uses Ore's plain-text name. Call `set_latex_symbols()` to restore the display defaults.

In [4]:
print(operator_latex(P, S="T"))
print(L)
set_latex_symbols()

T^{2} - T - 1
Dx - 1


## A sequence in $q$

Pass Laurent polynomials directly to `guess_qsequence`. For $f(n)=q^{n^2}$, let $Q f(n)=q^n f(n)$, so $EQ=qQE$. The degree bound here applies to $Q$.

In [5]:
q = LaurentPolynomialRing(QQ, "q").gen()
q_values = [q**(n*n) for n in range(24)]
q_guess = guess_qsequence(q_values, max_order=1, max_degree=2)
q_guess.to_ore()

-E + q*Q^2

This gives $f(n+1)=q^{2n+1} f(n)$ on the checked windows. The same sequence is used in Hologuess's [backend regression](../tests/test_flint_backend.py).

## Try your own data

Edit `values` in the first cell and adjust `order` and `degree`. Display `fibonacci_guess.operators` to see any returned operators. To see what an empty result looks like, try a first-order constant-coefficient search on the Fibonacci data.

In [6]:
guess(values, shift, order=1, degree=0).operators

()

Use `path=` to choose which pairs of bounds to try. Each pair gives the maximum order and maximum coefficient degree. Here `path=[(3, 0)]` tries just one pair, allowing constant coefficients on $1,S,S^2,S^3$. It finds two independent operators. Using `order=3, degree=0` would start at order one and stop when it finds the Fibonacci recurrence at order two. Use `.operators` to see the full basis for the selected bounds.

In [7]:
guess(values, shift, path=[(3, 0)]).operators

(Sn^2 - Sn - 1, Sn^3 - 2*Sn - 1)

## Check reserved observations

Apply the returned operator to the four checking windows above using its Sage coefficient list in order $1,E,E^2$. Zero residuals check those windows exactly. An all-index statement needs a separate argument.

In [8]:
operator = fibonacci_guess.operator
stop = len(values) - operator.order()
[sum(c(n) * values[n + j] for j, c in enumerate(operator.list()))
 for n in range(stop - 4, stop)]

[0, 0, 0, 0]

Explore recurrences with variable coefficients, algebraic and differential equations, and relations among several actions in the [worked examples](worked_examples.ipynb).